# P3 Project — YOLO 전이학습을 이용한 야생동물 자동 관측·분석 시스템

> **연구 질문:** **객체 탐지 성능(mAP)이 높은 모델이 종류별 출현 및 사진별 개체 수도 더 정확하게 집계할까?**

P3 Lab에서 배운 **CNN의 특성 추출 → Xception 전이학습/미세조정 → Faster R-CNN 객체 탐지**를 바탕으로, **YOLO11n 전이학습 → 관측 시스템 구현 → 탐지 성능과 실제 관측 통계의 신뢰성 검증**까지 진행한다.

**문제의 수준은 Lab보다 한 단계 높다.** 단순히 예측 상자를 그리는 것으로 끝나지 않는다. 여러 장의 정지 이미지에 대해 **객체별 탐지 기록과 사진별·종류별 집계, 출현 사진 비율, 오류 사례**를 생성하고, 서로 다른 학습 전략을 비교한다.

- **데이터:** African Wildlife — buffalo, elephant, rhino, zebra. 원본 분할(train/val/test)을 유지한다.
- **필수 실험:** 동일한 COCO 사전훈련 YOLO11n에서 출발하는 `frozen_10`, `full_0` 두 실험. `partial_5`는 선택 실험.
- **최종 산출물:** 실행한 노트북, 학습 가중치(직접 학습한 경우), 검증 실험표, 최종 테스트 결과, 사진/객체별 CSV, 관측 통계와 시각화, 오류 분석, AI 활용·검증 기록.
- **관측 대상:** 사진에 **보이는 객체**와 **객체가 등장한 사진**. 여러 사진의 객체 수 합은 실제 개체군 규모가 아니다. 촬영 시각·위치가 없으므로 시간대별 행동·지역별 분포·개체 이동 경로를 추론하지 않는다.

**표기:** `TODO ★`는 필수 학생 작성, `TODO ☆`는 선택 도전이다. **도움을 받아도 최종 코드를 실행·검증·설명할 책임은 학생에게 있다.**

## 프로젝트의 작업 흐름과 평가 기준

1. **정답 데이터 탐색:** 이미지별 YOLO 정답 박스 좌표를 픽셀 좌표로 변환하고, 종류별 **객체 수**와 **출현 사진 수**를 구분한다.
2. **사전훈련 모델 확인:** COCO 사전훈련 YOLO의 탐지 예시를 관찰한다. **COCO 클래스 체계와 이 프로젝트의 4개 클래스는 다르므로** 이 상태의 mAP를 4종 학습 모델과 직접 비교하지 않는다.
3. **전이학습 실험:** 동결 전략만 달리하고 데이터 분할·초기 가중치·이미지 크기·에포크 상한·학습률·시드를 고정한다.
4. **검증 성능 비교:** 모든 학습 모델의 mAP50-95 **및 공통 신뢰도 임곗값에서의 Count MAE**를 비교한다. 이 과정으로 핵심 연구 질문에 답한다.
5. **모델·임곗값 확정:** 사전에 선언한 규칙대로 검증 mAP50-95 1위 모델을 선정하고, 이 모델의 검증 Count MAE로 임곗값을 결정한다. 연구 질문 분석용 비교와 최종 모델 선정 절차를 구분한다.
6. **최종 테스트 및 적용:** 선택을 끝낸 뒤 테스트셋을 **한 번** 평가하고, 관측 보고서·오류 사례·새 사진 분석 결과를 작성한다.

### 채점에서 중요한 것
- 단순히 높은 mAP를 얻었는지가 아니라 **통제된 실험·올바른 지표·검증된 코드·데이터에 근거한 해석**을 보겠다.
- `TODO ★`를 직접 완성하고 테스트를 통과시킨다. AI 코드·설명은 검증 없이 신뢰하지 않는다.
- **미실행 결과를 만들어 넣지 않는다.** 모델을 제공받아 사용했다면 직접 학습한 것처럼 보고하지 않는다.

### 안내
아래 전체 파이프라인은 의도적으로 실행 가능한 **공통 지원 코드**와 **핵심 계산 TODO**로 구성된다. 코드를 단순히 줄이는 것이 목표는 아니다. 실제로 비교 가능한 실험을 하되, 복잡한 반복 처리·저장·학습 엔진은 라이브러리에 맡긴다.

## AI 활용 규칙: 설명 요청 → 코드 작성 → 테스트 → 결과 검증

생성형 AI를 사용해도 된다. 단, 아래 작업 기록을 **최소 3개** 남긴다.

| 기록 | 학생이 제출할 내용 |
|---|---|
| 질문/목표 | 무엇이 막혔고 AI에 무엇을 물었는가? |
| AI 제안 | 유용한 핵심 설명이나 코드 아이디어 (전체 대화 붙여넣기 불필요) |
| 내가 수행한 검증 | 단위 테스트, 이미지·라벨 대조, 계산 확인, 문서·출력 확인 중 무엇을 했는가? |
| 판단 및 수정 | AI 답을 그대로 썼는가, 수정했는가, 그 이유는 무엇인가? |

**권장 질문 예시:** ‘정답을 바로 주기보다 `boxes.cls` 자료형과 활용법을 설명해 줘. 내가 직접 완성한 뒤 테스트할 수 있게 검증 사례도 제안해 줘.’

**중요:** 모델의 코드가 오류 없이 실행된다는 사실은 탐지 결과나 관측 통계가 정확하다는 뜻이 아니다. 검증셋은 실험 설계·모델·임곗값 선택에, 테스트셋은 최종 평가에만 사용한다.

**실행 환경:** Ultralytics YOLO(PyTorch 기반). 초기 모델과 데이터셋은 인터넷 다운로드가 필요하다. GPU가 부족하면 **학습용 공통 제공 체크포인트**를 사용해 평가·응용 단계를 수행할 수 있으나, 제공 가중치의 출처와 직접 수행 범위를 명시해야 한다.

### 강의노트와의 연결 (P3 Lab → P3 Project)

| P3 Lab에서 학습한 내용 | 프로젝트에서의 활용 | 평가에 필요한 설명 |
|---|---|---|
| CNN 특성 추출·이미지 분류 | 객체별 클래스 예측의 기반 | 이미지 전체 분류와 객체별 클래스+위치 예측의 차이 |
| Xception 전이학습·미세 조정 | YOLO 사전훈련 가중치 재사용, 동결 범위 변경 | 동결 범위·데이터 유사성·검증 성능의 관계 |
| Faster R-CNN 추론 | YOLO의 클래스·점수·박스 해석 | `labels`/`scores`/`boxes`와 `boxes.cls`/`conf`/`xyxy` 연결 |
| 신뢰도 임곗값 0.30/0.80 비교 | Confidence Threshold에 따른 개체 수·오탐·미탐 변화 | 높은 신뢰도 ≠ 무조건 정확한 관측 수 |
| 훈련/검증/테스트 분리 | 실험·임곗값 선택 / 최종 평가 구분 | 테스트를 반복 선택에 쓰면 안 되는 이유 |

**학습 전략:** 복잡한 YOLO 내부 구현과 파일 저장용 반복문은 제공한다. 학생은 좌표 변환, 클래스별 집계, Count MAE와 모델 비교를 구현·검증한다.

---
## 1. 실행 환경과 재현성

Lab의 PyTorch CNN 실습과 동일하게 **장치(CPU/GPU), 초기조건, 학습/검증 모드**를 구분한다. 모델을 처음 불러오거나 데이터셋을 다운로드할 때 인터넷이 필요하다.

필수 라이브러리: `ultralytics`, `torch`, `pandas`, `numpy`, `matplotlib`, `Pillow`, `scikit-learn`.

별도 셀에서 처음 한 번 실행: `%pip install ultralytics pandas matplotlib pillow scikit-learn`

**실행 안내:** 아래의 `RUN_*` 스위치는 기본적으로 `False`다. 무거운 실험·최종 테스트를 시작하기 전에 직접 켜야 한다. 실행한 뒤 실험 설정과 로그를 반드시 저장한다.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path.cwd() / "P3_wildlife_project"
RUNS_DIR = PROJECT_DIR / "runs"
OUTPUT_DIR = PROJECT_DIR / "outputs"
for folder in [PROJECT_DIR, RUNS_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_YAML = "african-wildlife.yaml"
MODEL_NAME = "yolo11n.pt"  # 공개 사전훈련 경량 탐지 모델
DEVICE = "0" if torch.cuda.is_available() else "cpu"
IMGSZ = 512
BATCH = 8 if torch.cuda.is_available() else 4
EPOCHS = 25
INITIAL_LR = 0.001

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Project directory:", PROJECT_DIR)

### 실행 스위치와 학습 계획

- **필수 A:** `frozen_10` — 모델 앞쪽 10개 층을 동결한 전이학습.
- **필수 B:** `full_0` — 모든 층을 학습 대상으로 두는 전체 미세조정.
- **선택 C:** `partial_5` — 앞쪽 5개 층만 동결.
- 모두 **동일한 사전훈련 초기 가중치**에서 시작한다. 강의의 Xception `trainable=False` / 일부 층 해제와 연결해 설명하자.

학습 여건이 부족하면 교수자가 제공하는 `runs/<실험명>/weights/best.pt`를 프로젝트 폴더의 동일한 위치에 넣고, **검증·집계·테스트·오류 분석**을 수행할 수 있다. 제공 가중치를 사용한 경우 보고서에 표시한다.

**TODO ★ (설계):** 학습 시작 전에 모델 A/B에서 바꾸는 요소와 고정해야 하는 요소를 표로 작성하라. 결과가 더 좋은 쪽을 가정하고 그 이유도 적는다.

> 고정 요소:  
> 변경 요소:  
> 예상:  
> 예상 근거 (CNN/Xception Lab과 연결):

In [ ]:
RUN_TRAINING = {
    "frozen_10": False,  # 필요할 때 True
    "full_0": False,
    "partial_5": False,  # 선택
}
RUN_VALIDATION = False          # 두 모델의 검증 mAP
RUN_COMPARE_COUNTS = False      # ★ 두 모델의 검증 Count MAE 비교
RUN_COUNT_THRESHOLD = False     # 검증셋에서 최종 임곗값 결정
RUN_FINAL_TEST = False          # 검증 결정 후 한 번만 실행

EXPERIMENTS = {
    "frozen_10": 10,
    "full_0": 0,
    "partial_5": 5,
}
COUNT_THRESHOLDS = [0.20, 0.35, 0.50, 0.65]
# 모델들 사이의 공정한 Count MAE 비교에 사용할 단일 공통 임곗값
COMPARISON_THRESHOLD = 0.35

---
## 2. 데이터 탐색 — 종류, 위치, 객체 수와 출현 사진 수

African Wildlife는 buffalo, elephant, rhino, zebra의 객체 탐지 데이터셋이다. 1,504장의 원본 이미지를 **train 1,052 / val 225 / test 227**로 사용한다. 수량은 다운로드 후 코드로 검증한다.

정답 주석은 각 객체에 대한 `(class_id, x_center, y_center, width, height)` 형태의 **정규화된 YOLO 박스**다. Lab의 Faster R-CNN 예측 `(x1,y1,x2,y2)` **픽셀 좌표**와 다르다. **TODO ★**에서는 두 좌표계 사이의 변환을 직접 완성한다.

In [ ]:
data_info = check_det_dataset(DATA_YAML)  # 최초 실행 시 자동 다운로드
DATA_ROOT = Path(data_info["path"])
CLASS_NAMES = [data_info["names"][i] for i in range(4)]

IMAGE_DIRS = {split: Path(data_info[split]) for split in ["train", "val", "test"]}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def collect_images(directory):
    return sorted(p for p in directory.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)

image_paths = {split: collect_images(folder) for split, folder in IMAGE_DIRS.items()}

print("Classes:", CLASS_NAMES)
for split in ["train", "val", "test"]:
    print(f"{split:>5s}: {len(image_paths[split]):4d} images — {IMAGE_DIRS[split]}")
print("Total images:", sum(len(v) for v in image_paths.values()))
assert sum(len(v) for v in image_paths.values()) == 1504, "데이터 개수를 확인하세요."

### TODO ★ 1. 정규화된 YOLO 경계 상자를 픽셀 좌표로 변환

강의의 Faster R-CNN 시각화는 `(x1, y1, x2, y2)` 픽셀 좌표를 사용했다. 여기서는 YOLO 정답 `(xc, yc, w, h)`가 **0~1 사이 값**으로 주어진다.

- 이미지 너비 `W`, 높이 `H`라 할 때, 중심·폭·높이를 픽셀 단위로 환산한다.
- 그다음 왼쪽 위와 오른쪽 아래 좌표를 계산한다.
- 제공된 테스트는 **실제 이미지가 아닌 계산 점검용**이다. 테스트가 통과하면 훈련 이미지에 적용한다.

힌트: `x1 = (xc - bw/2)*W`, `y2=(yc+bh/2)*H`. **좌표 순서**를 주의하라.

**AI 활용 기록 후보:** ‘YOLO 라벨 좌표와 Faster R-CNN의 픽셀 좌표가 어떻게 다른가?’

In [ ]:
def label_path_for(image_path):
    # African Wildlife 구조: images/{split} ↔ labels/{split}
    return DATA_ROOT / "labels" / image_path.parent.name / f"{image_path.stem}.txt"


def read_annotations(image_path):
    label_path = label_path_for(image_path)
    if not label_path.exists():
        return []  # 라벨 누락과 객체가 없는 이미지를 따로 확인할 것
    labels = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        values = line.split()
        if len(values) != 5:
            raise ValueError(f"Invalid annotation: {label_path} — {line}")
        class_id = int(values[0])
        xywh = [float(v) for v in values[1:]]
        labels.append((class_id, *xywh))
    return labels


def yolo_box_to_pixels(xc, yc, bw, bh, width, height):
    """TODO ★ 1: YOLO 정규화 좌표 → (x1, y1, x2, y2) 픽셀 좌표."""
    # TODO ★: 아래 4개 좌표값을 계산하여 반환
    raise NotImplementedError("TODO ★ 1: YOLO 박스 좌표 변환을 구현하세요")
    # return x1, y1, x2, y2


# 먼저 테스트한 뒤 아래 시각화를 실행하라.
test_box = yolo_box_to_pixels(0.5, 0.5, 0.4, 0.2, 200, 100)
assert np.allclose(test_box, (60, 40, 140, 60)), test_box
print("TODO ★ 1 passed")


def draw_true_boxes(ax, image_path):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    ax.imshow(image)
    for class_id, xc, yc, bw, bh in read_annotations(image_path):
        x1, y1, x2, y2 = yolo_box_to_pixels(xc, yc, bw, bh, width, height)
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               edgecolor="yellow", facecolor="none", linewidth=2))
        ax.text(x1, max(y1 - 4, 0), CLASS_NAMES[class_id], color="black", fontsize=8,
                bbox=dict(facecolor="yellow", alpha=0.85, edgecolor="none"))
    ax.axis("off")


samples = random.Random(SEED).sample(image_paths["train"], 6)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, path in zip(axes.flat, samples):
    draw_true_boxes(ax, path)
    ax.set_title(f"{path.stem[:22]} | GT: {len(read_annotations(path))}", fontsize=10)
plt.tight_layout()

### TODO ★ 2. ‘탐지된 객체 수’와 ‘해당 종류가 있는 사진 수’ 구분

객체 탐지 라벨에 `[3,3,0]`이 들어 있는 사진 1장의 경우 얼룩말 정답 객체 수는 **2**, 얼룩말 출현 사진 수는 **1**이다.

`Counter.update(ids)`는 같은 값의 등장 횟수를 센다. `set(ids)`를 활용하면 한 사진에서 같은 종류가 여러 번 나와도 출현 사진 수를 한 번만 셀 수 있다.

**TODO ★:** 다음 코드는 훈련셋 전체에서 (1) 클래스별 박스 수, (2) 클래스별 출현 사진 수를 집계한다. 두 줄을 직접 작성하고 그래프로 비교한다.

In [ ]:
gt_box_counts = Counter()
gt_image_presence = Counter()
for path in image_paths["train"]:
    ids = [row[0] for row in read_annotations(path)]
    # TODO ★ 2: ids로 정답 객체 수 누적
    raise NotImplementedError("TODO ★ 2: gt_box_counts 갱신을 구현하세요")
    # TODO ★ 2: ids에서 중복 종류를 제거하여 출현 사진 수 누적
    # (위의 raise를 삭제하고 두 개의 실제 코드를 채워 넣으세요)

train_summary = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Annotated boxes": [gt_box_counts[i] for i in range(4)],
    "Images containing class": [gt_image_presence[i] for i in range(4)],
})
assert (train_summary["Annotated boxes"] >= train_summary["Images containing class"]).all()
assert (train_summary["Images containing class"] <= len(image_paths["train"])).all()
display(train_summary)

ax = train_summary.plot.bar(x="Class", y=["Annotated boxes", "Images containing class"],
                            figsize=(9, 4), rot=0)
ax.set_title("Training annotations")
ax.set_ylabel("Count")
plt.tight_layout()

### 데이터 이해 — 직접 답하기 (필수)

1. 강의의 이미지 분류 문제와 달리, 한 사진에서 **여러 클래스와 여러 경계 상자**를 예측해야 하는 이유는 무엇인가?
2. `images/train`과 `labels/train` 파일명은 어떻게 연결되는가? 정규화 좌표와 픽셀 좌표는 어떻게 다른가?
3. 훈련셋에서 **객체 수가 많지만 출현 사진 수가 상대적으로 적은** 클래스가 있다면 무엇을 의미하는가?
4. `read_annotations()`가 없는 파일을 `[]`로 반환할 때 어떤 자료 품질 문제가 가려질 수 있을까?
5. 사진에 세어진 총 객체 수가 실제 개체군의 크기가 아닌 이유는 무엇인가?

**답 (실제 출력·그림에 근거하여 작성):**

> TODO ★

---
## 3. COCO 사전훈련 YOLO와 동결 범위 확인

P3 Lab에서는 COCO 사전훈련 Faster R-CNN을 **추론에만** 이용했다. 이번 프로젝트에서는 **YOLO11n을 새로운 4개 클래스에 맞게 학습**한다. Xception 전이학습에서 해보았던 `trainable=False`와 일부 층만 동결 해제하는 전략을 연결해 이해하자.

- **사전훈련 그대로 사용 (정성적 기준선):** 예측 상자·클래스·신뢰도를 **관찰**한다.
- **A (`frozen_10`), B (`full_0`), C (`partial_5`):** 4개 클래스에 맞춰 학습한 모델끼리 **같은 검증셋의 mAP**를 비교한다.

**비교의 주의:** COCO와 African Wildlife의 클래스 정의·인덱스가 같지 않다. 따라서 사전훈련 COCO 모델에서 나온 예측을 4클래스 정답과 바로 비교하여 `mAP50-95`를 계산하는 것은 타당하지 않다. 정성적 관찰과 정량적 모델 비교를 구분한다.

In [ ]:
initial_model = YOLO(MODEL_NAME)
for idx, layer in enumerate(initial_model.model.model):
    marker = "frozen_10" if idx < 10 else "trainable (in experiment A)"
    print(f"{idx:2d} | {layer.__class__.__name__:22s} | {marker}")

### 사전훈련 COCO 모델의 **정성적 기준선** 관찰

P3 Lab Faster R-CNN처럼, 학습하지 않은 `initial_model`로 훈련셋의 일부 이미지를 **관찰용으로** 예측한다. COCO 클래스와 African Wildlife 네 클래스를 임의로 같은 인덱스로 해석하지 않는다.

**TODO ★:** 네 종류 중 COCO에서 이름이 직접 대응되는 종류와 그렇지 않은 종류를 구별하라. `initial_model.names`를 출력하여 실제 레이블을 확인한다. 관찰 결과와 4클래스 모델의 mAP를 직접 비교하지 않는다.

In [ ]:
# 사전훈련 COCO 예측은 정량적 모델 비교용이 아닌 시각적 관찰용
RUN_COCO_DEMO = False  # 첫 모델이 준비되고 CPU/GPU 상황이 허용되면 True
if RUN_COCO_DEMO:
    print("COCO classes (name examples):", list(initial_model.names.items())[:12])
    for path in samples[:2]:
        result = initial_model.predict(str(path), imgsz=IMGSZ, conf=0.30,
                                       device=DEVICE, verbose=False)[0]
        plt.figure(figsize=(7, 5))
        plt.imshow(result.plot()[:, :, ::-1])
        plt.title(f"COCO pretrained prediction | {path.name}")
        plt.axis("off")
        plt.show()
else:
    print("Set RUN_COCO_DEMO=True for optional qualitative baseline.")

### TODO ★ 3. 모델 실험 가설 먼저 작성하기

1. `freeze=10`과 `freeze=0`이 **같은 지점**과 **다른 지점**은 무엇인가? 둘 다 사전훈련 가중치를 사용하는가?
2. Xception Lab에서의 `base.trainable=False`, 일부 층 동결 해제와 이 프로젝트의 동결 범위를 연결해 설명하라.
3. **가설 A:** `freeze=10`과 `freeze=0` 중 어느 쪽이 검증 mAP50-95가 높을까? 이유는?
4. **가설 B:** mAP50-95가 높다고 사진별 종류별 Count MAE도 반드시 낮을까? 그 이유는?

**실험 전 예상과 근거:**

> TODO ★

**실험 후 예상과 달랐던 점:**

> TODO ★ (실행 후 작성)

---
## 4. 전이학습·미세조정 실험

**실험의 본질:** YOLO가 자동으로 훈련을 수행하지만, 어떤 층을 고정하고 무엇을 통제하며 어떤 결과를 비교해야 하는지는 **학생의 실험 설계 문제**다.

- `frozen_10`: 앞쪽 10개 층을 동결하고 나머지를 학습.
- `full_0`: 모든 층의 학습을 허용.
- `partial_5`: 앞쪽 5개 층 동결(선택).

동일 사전훈련 체크포인트에서 **매번 새 인스턴스**를 만들고, 초기 학습률·시드·배치 크기·데이터 분할·이미지 크기·에포크 상한을 공통 적용한다. 조기 종료로 **실제 학습 에포크 수**가 달라질 수 있으므로 함께 기록한다.

**실행 부담:** GPU가 없을 때 학습 시간은 상당할 수 있다. 강의 담당자가 제공한 가중치를 쓰는 경우 학습 실험의 직접 수행 여부를 구분하고 이후 분석을 동일하게 한다.

In [ ]:
def checkpoint_for(tag):
    return RUNS_DIR / tag / "weights" / "best.pt"

training_records = []
for tag, frozen_layers in EXPERIMENTS.items():
    if not RUN_TRAINING[tag]:
        continue

    print(f"Starting {tag} (freeze={frozen_layers})")
    model = YOLO(MODEL_NAME)  # 매 실험 같은 사전훈련 상태에서 시작
    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=0,              # Windows 개인 PC 기본 설정
        freeze=frozen_layers,
        optimizer="AdamW",
        lr0=INITIAL_LR,
        patience=8,
        seed=SEED,
        deterministic=True,
        fraction=1.0,
        project=str(RUNS_DIR),
        name=tag,
        exist_ok=True,
        plots=True,
    )
    output_dir = Path(model.trainer.save_dir)
    best_path = output_dir / "weights" / "best.pt"
    assert best_path.exists(), f"Checkpoint not found: {best_path}"
    training_records.append({"experiment": tag, "freeze": frozen_layers,
                             "checkpoint": str(best_path)})

if not training_records:
    print("Training is disabled. Enable the desired RUN_TRAINING values to start experiments.")
else:
    display(pd.DataFrame(training_records))

### 훈련 로그 읽기

학습한 각 실험의 `runs/{실험명}/results.csv`를 활용해 훈련 경향을 확인한다. 탐지 모델에는 여러 손실이 있으므로 **mAP의 변화**와 함께 읽자.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
found_curves = False
for tag in EXPERIMENTS:
    csv_path = RUNS_DIR / tag / "results.csv"
    if not csv_path.exists():
        continue
    history = pd.read_csv(csv_path)
    history.columns = [col.strip() for col in history.columns]
    metric_col = "metrics/mAP50-95(B)"
    if metric_col not in history:
        print(f"Missing column in {tag}: {metric_col}")
        continue
    epoch_numbers = history["epoch"]
    if epoch_numbers.min() == 0:
        epoch_numbers = epoch_numbers + 1
    ax.plot(epoch_numbers, history[metric_col], marker=".", label=tag)
    found_curves = True
if found_curves:
    ax.set(xlabel="Epoch", ylabel="mAP50-95", title="Validation detection performance")
    ax.legend()
else:
    ax.text(0.5, 0.5, "Run training experiments first", ha="center", va="center", transform=ax.transAxes)
    ax.axis("off")
plt.tight_layout()

**TODO ★ (학습곡선 해석):** 두 모델의 검증 mAP 변화가 어느 epoch에서 완만해지는지 비교하라. *실제 로그가 없는 경우에는 이 셀을 근거로 결과를 추정하지 말 것.*

### 검증셋 mAP로 탐지 성능 비교

검증 **mAP50**와 **mAP50-95**는 객체의 **클래스·위치 일치**를 반영한다. 사진별 개체 수 오차와는 평가 목적이 다르다. 비교 모델은 동일한 검증 데이터와 같은 지표를 사용한다.

`RUN_VALIDATION=True`로 저장한 모델별 mAP를 계산하라. **TODO ★:** 이후 나타날 Count MAE 비교표와 함께 해석해야 한다. mAP만으로 핵심 연구 질문에 답하지 않는다.

In [ ]:
val_rows = []
if RUN_VALIDATION:
    for tag in EXPERIMENTS:
        best = checkpoint_for(tag)
        if not best.exists():
            continue
        model = YOLO(str(best))
        metric = model.val(data=DATA_YAML, split="val", imgsz=IMGSZ,
                           batch=BATCH, device=DEVICE, plots=False, verbose=False)
        val_rows.append({
            "experiment": tag,
            "freeze": EXPERIMENTS[tag],
            "mAP50": float(metric.box.map50),
            "mAP50-95": float(metric.box.map),
            "checkpoint": str(best),
        })
    if val_rows:
        pd.DataFrame(val_rows).to_csv(OUTPUT_DIR / "validation_models.csv", index=False)

val_csv = OUTPUT_DIR / "validation_models.csv"
if val_csv.exists():
    val_table = pd.read_csv(val_csv).sort_values("mAP50-95", ascending=False)
    display(val_table.drop(columns=["checkpoint"]))
else:
    print("Run the training and validation cells, then set RUN_VALIDATION=True.")

### TODO ★ 4. 전이학습 결과 해석

1. 실험 A/B에서 **유일하게 의도적으로 바꾼 조건**은 무엇인가? 실제 에포크 수, 학습률과 주요 설정을 기록하라.
2. 검증 mAP50-95가 더 높은 모델은? 얼마나 차이나는가?
3. 동결한 층이 더 적으면 항상 성능이 높아지는가? 이 두 실험이 그 일반 명제를 증명하는가?
4. 학습 곡선과 검증 mAP에 근거하여 과적합 또는 학습 부족의 가능성을 논의하라.
5. **이 시점에서 개체 수 집계 성능까지 우수하다고 말할 수 있는가?** 아직 부족한 분석은 무엇인가?

| 모델 | freeze | 실제 epoch | val mAP50 | val mAP50-95 | 훈련곡선·결과 해석 |
|---|---:|---:|---:|---:|---|
| A: frozen_10 | 10 | | | | |
| B: full_0 | 0 | | | | |
| C: partial_5 (선택) | 5 | | | | |

**답:**

> TODO ★

---
## 5. 시스템의 핵심 — 객체별 탐지 기록에서 사진별 개체 수로

Lab의 Faster R-CNN 추론에서는 `boxes`, `labels`, `scores`를 읽었다. Ultralytics YOLO의 각 예측에서는 `result.boxes.xyxy`, `result.boxes.cls`, `result.boxes.conf`를 읽는다.

- **객체별 레코드:** 이미지명, 클래스, 신뢰도, 박스 좌표
- **사진별 레코드:** 이미지명, 클래스별 실제 개체 수·예측 개체 수, 총 개체 수

시스템 처리 루프, CSV 생성은 **제공 코드**다. 그러나 종류별 개수를 세는 함수와 **Count MAE 계산은 학생이 구현**한다. 단위 테스트를 통과하지 못한 상태에서 실제 실험 수치를 해석하지 않는다.

**주의:** 같은 객체에 중복 박스가 생기거나 다른 배경을 잘못 탐지하면 집계가 과대평가된다. **박스 수가 같다고 모든 객체를 정확히 찾은 것은 아니다.**

In [ ]:
def count_class_ids(class_ids, n_classes):
    """TODO ★ 5: 클래스 번호 목록을 받아 클래스별 개수(길이 n_classes 리스트)를 반환."""
    # 예: [0, 3, 3], n_classes=4  -> [1, 0, 0, 2]
    raise NotImplementedError("TODO ★ 5: 클래스별 개수 계산 함수를 구현하세요")


# 데이터 없이 수행하는 기본 테스트
assert count_class_ids([0, 3, 3], 4) == [1, 0, 0, 2]
assert count_class_ids([], 4) == [0, 0, 0, 0]
assert sum(count_class_ids([1, 1, 2], 4)) == 3
print("TODO ★ 5 passed")


def monitor_images(model, paths, threshold=0.35, batch=8):
    """한 분할의 모든 사진에 대해 예측하고 사진/객체별 테이블 생성 (지원 코드)."""
    if not paths:
        raise ValueError("No images provided")
    parents = {p.parent.resolve() for p in paths}
    if len(parents) != 1:
        raise ValueError("All monitored images must be in one split directory")
    source_dir = next(iter(parents))
    if {p.resolve() for p in paths} != set(collect_images(source_dir)):
        raise ValueError("Pass the entire split directory without sampling")
    results = model.predict(source=str(source_dir), stream=True, conf=threshold,
                            imgsz=IMGSZ, batch=batch, device=DEVICE, verbose=False)
    image_records, object_records = [], []
    for result in results:
        image_path = Path(result.path)
        predicted = [int(v) for v in result.boxes.cls.cpu().numpy()]
        gt_classes = [row[0] for row in read_annotations(image_path)]
        pred_counts = count_class_ids(predicted, len(CLASS_NAMES))
        true_counts = count_class_ids(gt_classes, len(CLASS_NAMES))
        rec = {"image": image_path.name,
               "true_total": len(gt_classes), "pred_total": len(predicted)}
        for class_id, cls_name in enumerate(CLASS_NAMES):
            rec[f"true_{cls_name}"] = true_counts[class_id]
            rec[f"pred_{cls_name}"] = pred_counts[class_id]
        image_records.append(rec)
        coords = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        for cls_id, score, xyxy in zip(predicted, scores, coords):
            object_records.append({
                "image": image_path.name, "class": CLASS_NAMES[cls_id],
                "confidence": float(score),
                "x1": float(xyxy[0]), "y1": float(xyxy[1]),
                "x2": float(xyxy[2]), "y2": float(xyxy[3]),
            })
    if len(image_records) != len(paths):
        raise RuntimeError(f"Only {len(image_records)} / {len(paths)} images were processed")
    images_df = pd.DataFrame(image_records).sort_values("image").reset_index(drop=True)
    objects_df = pd.DataFrame(object_records, columns=[
        "image", "class", "confidence", "x1", "y1", "x2", "y2"])
    return images_df, objects_df


def count_error_report(images_df):
    """TODO ★ 6: 클래스별 Count MAE와 이미지별 절대오차 합의 평균을 반환."""
    y_true = images_df[[f"true_{c}" for c in CLASS_NAMES]].to_numpy()
    y_pred = images_df[[f"pred_{c}" for c in CLASS_NAMES]].to_numpy()
    # TODO ★: (1) y_true와 y_pred의 원소별 절대오차를 계산
    # TODO ★: (2) 클래스별 평균 절대오차 (열별 평균)
    # TODO ★: (3) 이미지별 네 클래스 절대오차의 합을 구한 뒤 이미지별 평균
    raise NotImplementedError("TODO ★ 6: Count MAE와 이미지별 절대오차를 구현하세요")
    # return pd.DataFrame({"Class": CLASS_NAMES, "Count MAE": per_class_mae}), float(image_sum_mae)


# 학생 코드의 단위 테스트: 아래 수치는 실제 실험 결과가 아니다.
example = pd.DataFrame({
    "true_buffalo": [2, 0], "true_elephant": [0, 1],
    "true_rhino": [0, 0], "true_zebra": [1, 0],
    "pred_buffalo": [1, 0], "pred_elephant": [0, 1],
    "pred_rhino": [0, 0], "pred_zebra": [2, 0],
})
example_by_class, example_total = count_error_report(example)
assert np.allclose(example_by_class["Count MAE"].to_numpy(), [0.5, 0, 0, 0.5])
assert np.isclose(example_total, 1.0)
assert np.isclose(example_by_class["Count MAE"].mean(), example_total / len(CLASS_NAMES))
print("TODO ★ 6 passed: example by-class and image error")
display(example_by_class)

### 5-1. 핵심 연구 질문 분석 — **모든 모델의 개체 수 오차를 먼저 비교**

`RUN_COMPARE_COUNTS=True`로 설정하고, **훈련 완료한 모든 모델**에 대해 동일한 검증셋 **전체 사진**에서 `COMPARISON_THRESHOLD=0.35`를 고정하여 사진별 예측 개체 수를 구한다.

\[
\mathrm{CountMAE}=\frac{1}{N\,C}\sum_{i=1}^{N}\sum_{c=1}^{C}\left|\hat n_{ic}-n_{ic}\right|
\]

- `N`: 검증 이미지 수, `C=4`: 클래스 수. 낮을수록 좋다.
- 이 단계의 **공통 임곗값** 비교는 모델 A/B의 차이를 확인하기 위한 것이다. 모델별 임곗값을 따로 최적화한 수치와 섞어 해석하지 않는다.
- `mAP50-95`는 박스 위치·클래스 탐지를, `Count MAE`는 **사진별 클래스 개수**의 오차를 평가한다. 두 지표의 순위가 다른지 직접 확인한다.

**TODO ★ (해석):** 모델별 mAP와 Count MAE가 같은 순서로 좋아지는가? 2~3개 모델 결과만으로 보편적인 상관관계나 인과관계를 주장할 수 있는가?

**재현성:** 실험 설정이나 체크포인트를 바꿨다면 기존 `validation_models.csv`, `validation_count_by_model.csv`, `validation_thresholds.csv`, `model_selection.json`을 그대로 재사용하지 말고 다시 계산한다.

In [ ]:
# 같은 검증 데이터, 같은 공통 confidence 임곗값으로 모든 모델의 Count MAE를 비교한다.
# TODO ★ 5~6 함수가 정확히 작동해야 이 셀을 실행할 수 있다.
comparison_path = OUTPUT_DIR / "validation_count_by_model.csv"
compare_rows = []
if RUN_COMPARE_COUNTS:
    val_csv = OUTPUT_DIR / "validation_models.csv"
    if not val_csv.exists():
        raise FileNotFoundError("First set RUN_VALIDATION=True and evaluate trained models")
    val_metrics = pd.read_csv(val_csv)
    if len(val_metrics) < 2:
        raise ValueError("At least two trained models are required for the core comparison")
    for _, row in val_metrics.iterrows():
        tag = str(row["experiment"])
        checkpoint = checkpoint_for(tag)
        if not checkpoint.exists():
            raise FileNotFoundError(checkpoint)
        model = YOLO(str(checkpoint))
        images_df, _ = monitor_images(model, image_paths["val"],
                                      threshold=COMPARISON_THRESHOLD, batch=BATCH)
        class_mae, sum_mae = count_error_report(images_df)
        compare_rows.append({
            "experiment": tag, "freeze": EXPERIMENTS[tag],
            "mAP50-95": float(row["mAP50-95"]),
            "common threshold": COMPARISON_THRESHOLD,
            "Mean class Count MAE": float(class_mae["Count MAE"].mean()),
            "Mean image absolute count error": float(sum_mae),
        })
    compare_df = pd.DataFrame(compare_rows).sort_values(
        "mAP50-95", ascending=False).reset_index(drop=True)
    compare_df.to_csv(comparison_path, index=False)
if comparison_path.exists():
    compare_df = pd.read_csv(comparison_path)
    display(compare_df)
    ax = compare_df.plot(x="experiment", y="Mean class Count MAE",
                         kind="bar", rot=0, legend=False, figsize=(7, 4))
    ax.set(ylabel="Count MAE (lower is better)",
           title=f"Validation count error @ conf={COMPARISON_THRESHOLD}")
    plt.tight_layout()
else:
    print("Complete TODO ★ 5~6 and set RUN_COMPARE_COUNTS=True to compare models.")

### 5-2. 검증셋에서 **최종 모델과 신뢰도 임곗값 선택**

아래 스위치 `RUN_COUNT_THRESHOLD=True`를 켜면, 앞서 **검증 mAP50-95가 가장 높은 모델**을 먼저 선택한 후 해당 모델에 대해 4개의 임곗값으로 검증 Count MAE를 구한다.

**미리 정한 선택 규칙:** (1) 검증 mAP50-95 최대 모델, (2) 그 모델의 **사진·클래스별 Count MAE 최소** 임곗값, (3) 동점이면 큰 임곗값. 결과를 보고 규칙을 바꾸면 모델 선정이 임의적이 된다.

**중요:** 모델별 공통 임곗값에서의 Count MAE 비교가 **핵심 연구 질문**, 모델 선정 이후의 임곗값 탐색은 **실제 시스템 설정**이라는 역할을 각각 갖는다.

**TODO ★:** 최소 Count MAE를 주는 임곗값과 최종 선택 규칙의 의미를 실제 숫자로 설명하라.

In [ ]:
threshold_rows = []
selection_path = OUTPUT_DIR / "model_selection.json"
val_csv = OUTPUT_DIR / "validation_models.csv"

if RUN_COUNT_THRESHOLD:
    if not comparison_path.exists():
        raise FileNotFoundError("First compare at least two models on val Count MAE: RUN_COMPARE_COUNTS=True")
    if not val_csv.exists():
        raise FileNotFoundError("First run validation and create validation_models.csv")
    val_table = pd.read_csv(val_csv).sort_values("mAP50-95", ascending=False)
    chosen_tag = str(val_table.iloc[0]["experiment"])
    chosen_model = YOLO(str(checkpoint_for(chosen_tag)))
    for threshold in COUNT_THRESHOLDS:
        image_table, _ = monitor_images(chosen_model, image_paths["val"], threshold=threshold, batch=BATCH)
        class_mae, image_sum_mae = count_error_report(image_table)
        threshold_rows.append({
            "experiment": chosen_tag,
            "threshold": threshold,
            "Mean class Count MAE": float(class_mae["Count MAE"].mean()),
            "Mean image absolute count error": image_sum_mae,
        })
    thresholds_df = pd.DataFrame(threshold_rows).sort_values(
        ["Mean class Count MAE", "threshold"], ascending=[True, False]
    )
    thresholds_df.to_csv(OUTPUT_DIR / "validation_thresholds.csv", index=False)
    best_threshold = float(thresholds_df.iloc[0]["threshold"])
    selection = {"experiment": chosen_tag, "threshold": best_threshold,
                 "checkpoint": str(checkpoint_for(chosen_tag))}
    selection_path.write_text(json.dumps(selection, indent=2), encoding="utf-8")

threshold_csv = OUTPUT_DIR / "validation_thresholds.csv"
if threshold_csv.exists():
    display(pd.read_csv(threshold_csv))
else:
    print("Run model comparison and set RUN_COUNT_THRESHOLD=True.")
if selection_path.exists():
    print("Selected model and threshold:", json.loads(selection_path.read_text(encoding="utf-8")))

### 5-3. 두 성능 지표의 차이 — 직접 해석

1. 검증 mAP50-95와 Count MAE는 무엇을 평가하는가? (각각 **높을수록** 좋은지 **낮을수록** 좋은지도 표시)
2. 공통 임곗값 비교표에서 mAP 1위와 Count MAE 1위가 같은가? **숫자를 근거**로 답하라.
3. 같은 사진에 실제 동물 3마리·탐지 박스 3개가 있어도 실제 객체를 모두 맞게 탐지했다고 단정할 수 없는 이유는?
4. Confidence Threshold를 높이면 오탐과 미탐에 어떤 변화가 생길 수 있을까?
5. 모델을 선택할 때 왜 테스트셋의 성능을 미리 비교해서는 안 되는가?

**검증 결과에 근거한 답:**

> TODO ★

---
## 6. 최종 테스트 — 검증에서 확정한 모델·임곗값을 한 번 평가

**규칙:** 모델과 confidence threshold를 검증셋에서 확정한 뒤에만 `RUN_FINAL_TEST=True`로 바꾼다. 테스트셋은 성능 보고용이며 **결과에 맞춰 다시 학습하거나 설정을 바꾸면 독립 평가가 아니다**.

여기서는 **테스트 mAP50, mAP50-95, 종류별 Count MAE, 이미지별 네 종류의 절대오차 합 평균**을 구한다. 테스트 결과는 CSV로 저장하여 7~8절에서 **응용 시스템의 결과와 실패 사례**를 검토한다.

In [ ]:
if RUN_FINAL_TEST:
    if not selection_path.exists():
        raise FileNotFoundError("Select model and threshold using validation set first")
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    final_model = YOLO(selection["checkpoint"])
    final_threshold = float(selection["threshold"])

    # 한 번의 공식 테스트 평가: 위치·클래스·탐지 품질
    test_metrics = final_model.val(
        data=DATA_YAML, split="test", imgsz=IMGSZ,
        batch=BATCH, device=DEVICE, plots=True, verbose=False,
        project=str(RUNS_DIR), name="final_test_evaluation", exist_ok=True,
    )
    print("Test mAP50:", float(test_metrics.box.map50))
    print("Test mAP50-95:", float(test_metrics.box.map))

    test_images, test_detections = monitor_images(
        final_model, image_paths["test"], threshold=final_threshold, batch=BATCH
    )
    assert len(test_images) == 227
    test_images.to_csv(OUTPUT_DIR / "test_image_observations.csv", index=False)
    test_detections.to_csv(OUTPUT_DIR / "test_object_detections.csv", index=False)

    test_class_mae, test_image_sum_mae = count_error_report(test_images)
    test_class_mae.to_csv(OUTPUT_DIR / "test_count_mae_by_class.csv", index=False)
    print("Test image-wise absolute count error:", test_image_sum_mae)
    display(test_class_mae)
else:
    print("Final test is disabled. Select settings on validation, then set RUN_FINAL_TEST=True.")

---
## 7. 실제 관측 시스템 — 종류별 출현, 개체 수, 사진별 출현 비율

최종 테스트의 227장에 대해 다음을 구한다.

- **Pred boxes:** 여러 사진에서 탐지된 각 클래스 객체 수의 합계
- **Pred images:** 각 클래스가 한 마리 이상 탐지된 사진 수 (같은 사진에서 5마리여도 1장)
- **Pred image rate:** 해당 클래스가 탐지된 사진 수 / 전체 사진 수

**TODO ★ 7:** 이미 계산된 객체 수와 출현 사진 수를 활용하여 `Pred image rate`를 완성하라. 데이터에 없는 시간·위치 기반 패턴을 추론하지 말 것.

In [ ]:
observations_path = OUTPUT_DIR / "test_image_observations.csv"
if observations_path.exists():
    observations = pd.read_csv(observations_path)
    report_rows = []
    for c in CLASS_NAMES:
        n_pred_images = int((observations[f"pred_{c}"] > 0).sum())
        # TODO ★ 7: 해당 클래스가 관측된 사진의 비율
        # pred_image_rate = ...
        raise NotImplementedError("TODO ★ 7: pred_image_rate를 계산하세요")
        report_rows.append({
            "Class": c,
            "GT boxes": int(observations[f"true_{c}"].sum()),
            "Pred boxes": int(observations[f"pred_{c}"].sum()),
            "GT images": int((observations[f"true_{c}"] > 0).sum()),
            "Pred images": n_pred_images,
            "Pred image rate": pred_image_rate,
        })
    monitoring_summary = pd.DataFrame(report_rows)
    assert monitoring_summary["Pred image rate"].between(0, 1).all()
    monitoring_summary.to_csv(OUTPUT_DIR / "monitoring_summary.csv", index=False)
    display(monitoring_summary)
    ax = monitoring_summary.plot.bar(x="Class", y=["GT boxes", "Pred boxes"],
                                     rot=0, figsize=(9, 4))
    ax.set(title="Wildlife observations: object counts", ylabel="Count")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "wildlife_counts.png", dpi=160)
else:
    print("Run the final test to create observation reports.")

### 선택 심화 ☆: 두 동물이 **같은 사진에 나타난 빈도**

종류별 탐지 상자 수를 세는 데서 더 나아가, 두 종류가 같은 사진에서 나타난 횟수도 분석할 수 있다. 이는 **사진 단위 동시 출현**일 뿐 같은 장소/시간에서 실제로 만난 빈도나 행동 관계를 의미하지 않는다.

**TODO ☆:** `pred_...` 네 열을 0/1 출현 행렬로 바꿔서 `presence.T @ presence`로 클래스 쌍별 동시 출현 사진 수를 구하라. 대각 성분은 해당 클래스가 나온 사진 수와 같아야 한다.

In [ ]:
RUN_COOCCURRENCE = False  # 선택 ☆
if RUN_COOCCURRENCE:
    if not observations_path.exists():
        raise FileNotFoundError("Run final test first")
    observations = pd.read_csv(observations_path)
    cols = [f"pred_{name}" for name in CLASS_NAMES]
    # TODO ☆: 0보다 크면 1, 아니면 0인 N x 4 행렬 계산
    raise NotImplementedError("TODO ☆: presence matrix를 구현하세요")
    # TODO ☆: 클래스 쌍별 동시 출현 행렬 계산
    # cooccurrence = ...
    # display(pd.DataFrame(cooccurrence, index=CLASS_NAMES, columns=CLASS_NAMES))
else:
    print("Optional co-occurrence analysis disabled.")

### 7-1. 사진에 **등장하는지 여부**도 검증한다

P3 Lab의 분류 Precision/Recall과 연결한다. 각 이미지에서 해당 동물의 예측 개체 수가 하나 이상이면 `1`, 없으면 `0`으로 정의한 **사진 단위 출현 여부**를 평가한다.

아래 Precision, Recall, F1은 **박스 매칭에 기반한 객체 탐지 Precision/Recall이 아니라 사진 단위 출현 여부의 성능**이다. 모델 평가에서 어떤 단위를 대상으로 삼았는지 명확히 구분한다.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

if observations_path.exists():
    rows = []
    for c in CLASS_NAMES:
        truth = (observations[f"true_{c}"] > 0).astype(int)
        pred = (observations[f"pred_{c}"] > 0).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(
            truth, pred, average="binary", zero_division=0
        )
        rows.append({"Class": c, "Image precision": p,
                     "Image recall": r, "Image F1": f1})
    presence_df = pd.DataFrame(rows)
    presence_df.to_csv(OUTPUT_DIR / "test_presence_metrics.csv", index=False)
    display(presence_df)
else:
    print("Run the final test first.")

### 7-2. CSV에서 관측 보고서 작성하기 — TODO ★

1. `test_image_observations.csv` (한 이미지 1행)와 `test_object_detections.csv` (한 탐지 상자 1행)의 사용 목적을 구분하라.
2. `Pred boxes`와 `Pred images`가 크게 차이 나는 클래스가 있는가? 실제 이미지 1장을 제시하여 설명하라.
3. 출현 사진 비율은 단순 총 객체 수보다 어떤 경우에 더 유용할까?
4. 검증 결과에서 mAP가 높았더라도 출현 사진 F1이 낮을 수 있는가?
5. 촬영 위치·시각·반복 출현 객체 ID가 없는 데이터로 무엇을 말할 수 없을까?

**실제 CSV/그림을 인용한 답:**

> TODO ★

---
## 8. 오류 분석 — 수치로 찾고 사진으로 진단

종류별 예측 수가 실제 수와 다른 이미지를 **객체 개수 오차 합**으로 정렬하여 확인한다. 왼쪽은 정답 박스, 오른쪽은 예측 박스다. 단, 개체 수가 정확히 맞아도 **클래스·위치·동일 객체 중복 탐지**가 틀릴 수 있다.

단순한 점수 순위 제시보다, **세 장 이상의 서로 다른 오류 유형을 고르고 시각적 근거를 설명하는 것**이 핵심이다.

In [ ]:
if observations_path.exists() and selection_path.exists():
    obs = pd.read_csv(observations_path)
    errors = np.zeros(len(obs), dtype=int)
    for c in CLASS_NAMES:
        errors += (obs[f"true_{c}"] - obs[f"pred_{c}"]).abs().to_numpy()
    obs["total_abs_error"] = errors
    display(obs.sort_values("total_abs_error", ascending=False).head(8)[
        ["image", "true_total", "pred_total", "total_abs_error"]
    ])

    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    viz_model = YOLO(selection["checkpoint"])
    by_name = {p.name: p for p in image_paths["test"]}
    worst = obs.sort_values("total_abs_error", ascending=False).head(3)
    fig, axes = plt.subplots(len(worst), 2, figsize=(11, 4 * len(worst)))
    axes = np.atleast_2d(axes)
    for (i, (_, row)) in enumerate(worst.iterrows()):
        path = by_name[row["image"]]
        draw_true_boxes(axes[i, 0], path)
        axes[i, 0].set_title(f"Ground truth | {path.stem[:18]}")

        result = viz_model.predict(str(path), conf=float(selection["threshold"]),
                                   imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
        axes[i, 1].imshow(result.plot()[:, :, ::-1])  # BGR -> RGB
        axes[i, 1].set_title(f"Prediction | Count error: {row['total_abs_error']}")
        axes[i, 1].axis("off")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "error_examples.png", dpi=160)
else:
    print("Run the final test to inspect errors.")

### TODO ★ 8. 실패 사례 3장 이상 분석

적어도 다음 유형 중 **세 가지**를 이미지로 검증하고, 각 사례의 파일명·정답·예측·추정 원인을 기록한다.

- **미탐:** 가림, 작은 동물, 낮은 신뢰도 등으로 정답 객체가 누락되었는가?
- **오탐/클래스 오류:** 다른 동물·배경을 목표 동물로 예측했는가?
- **중복:** 동일 객체에 둘 이상의 박스가 남아 과대 집계했는가?
- **개수 일치하지만 오답:** 객체 수가 같아도 박스 위치 또는 클래스가 틀렸는가?

| 이미지 파일명 | 실제 / 예측 개체 수 | 오류 유형 | 이미지/점수 근거 | 현장 관측 시 영향 |
|---|---|---|---|---|
| (사례 1) | | | | |
| (사례 2) | | | | |
| (사례 3) | | | | |

**고찰:** Count MAE만으로 전체 객체 탐지 품질을 완전히 평가할 수 없는 이유는 무엇인가?

> TODO ★

---
## 9. 새 사진으로 관측 시스템 사용하기

`P3_wildlife_project/incoming_images/`에 관측 사진을 넣고 `RUN_NEW_PHOTOS=True`로 설정한다. 검증에서 확정한 모델과 임곗값으로 **종류별 동물 수, 객체별 좌표·신뢰도, 박스가 표시된 사진**을 생성한다.

**추가 응용의 목적:** 데이터셋 테스트 점수가 좋아도 새로운 환경에서는 성능이 달라질 수 있다. **새 사진은 정답 라벨이 없으므로 정량적 정확도를 검증한 것으로 보고하지 않는다.**

**TODO ★ 9:** 서로 다른 상황의 새 사진을 2장 이상 분석해 모델의 예측을 눈으로 검토하고 어떤 불확실성이 있는지 기록한다. 사진 확보가 어렵다면 검증·테스트에 사용하지 않은 사용권이 허용된 사진을 이용할 수 있다.

**선택 ☆:** 특정 동물 종류를 감지하면 알림용 CSV 행을 만드는 기능을 추가한다. 영상·시간 기반 추적은 **연속 영상 데이터가 별도로 있어야 하는 심화 과제**다.

In [ ]:
RUN_NEW_PHOTOS = False
NEW_IMAGE_DIR = PROJECT_DIR / "incoming_images"
ANNOTATED_DIR = OUTPUT_DIR / "annotated"
NEW_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

if RUN_NEW_PHOTOS:
    if not selection_path.exists():
        raise FileNotFoundError("Run validation model/threshold selection first")
    new_files = collect_images(NEW_IMAGE_DIR)
    if not new_files:
        print("Add image files to:", NEW_IMAGE_DIR)
    else:
        choice = json.loads(selection_path.read_text(encoding="utf-8"))
        deployed_model = YOLO(choice["checkpoint"])
        new_image_rows, new_object_rows = [], []
        for result in deployed_model.predict(
                source=str(NEW_IMAGE_DIR), stream=True, imgsz=IMGSZ,
                device=DEVICE, conf=float(choice["threshold"]), verbose=False):
            classes = [int(x) for x in result.boxes.cls.cpu().tolist()]
            scores = result.boxes.conf.cpu().tolist()
            boxes = result.boxes.xyxy.cpu().tolist()
            counts = Counter(classes)
            row = {"image": Path(result.path).name, "pred_total": len(classes)}
            row.update({f"pred_{name}": counts[idx] for idx, name in enumerate(CLASS_NAMES)})
            new_image_rows.append(row)
            for cls, score, xyxy in zip(classes, scores, boxes):
                new_object_rows.append({"image": Path(result.path).name,
                                        "class": CLASS_NAMES[cls], "confidence": score,
                                        "x1": xyxy[0], "y1": xyxy[1],
                                        "x2": xyxy[2], "y2": xyxy[3]})
            img_rgb = result.plot()[:, :, ::-1]
            Image.fromarray(img_rgb).save(ANNOTATED_DIR / Path(result.path).name)
        image_df = pd.DataFrame(new_image_rows)
        objects_df = pd.DataFrame(new_object_rows, columns=[
            "image", "class", "confidence", "x1", "y1", "x2", "y2"])
        image_df.to_csv(OUTPUT_DIR / "new_photo_observations.csv", index=False)
        objects_df.to_csv(OUTPUT_DIR / "new_photo_detections.csv", index=False)
        display(image_df.head())
        print("Saved annotated images to:", ANNOTATED_DIR)
else:
    print("New-photo analysis is disabled; set RUN_NEW_PHOTOS=True to run.")

### 선택 심화 ☆: 특정 종류 출현 알림 CSV 생성

새 사진 결과에서 `ALERT_CLASS`가 `ALERT_MIN_COUNT`마리 이상 탐지된 이미지만 골라 `new_photo_alerts.csv`를 만든다. **이 알림은 자동 탐지의 결과일 뿐 확인된 실제 출현 정보가 아니므로 오탐 확인이 필요**하다.

In [ ]:
RUN_ALERTS = False  # 선택 ☆: 반드시 새 사진 분석 후 실행
ALERT_CLASS = "elephant"  # CLASS_NAMES의 한 종류
ALERT_MIN_COUNT = 1
if RUN_ALERTS:
    new_csv = OUTPUT_DIR / "new_photo_observations.csv"
    if not new_csv.exists():
        raise FileNotFoundError("Run new-photo analysis first")
    if ALERT_CLASS not in CLASS_NAMES:
        raise ValueError(f"Choose a class from {CLASS_NAMES}")
    new_obs = pd.read_csv(new_csv)
    # TODO ☆: ALERT_CLASS가 ALERT_MIN_COUNT개 이상 탐지된 이미지만 선택
    raise NotImplementedError("TODO ☆: 알림 이미지 필터를 구현하세요")
    # alerts.to_csv(OUTPUT_DIR / "new_photo_alerts.csv", index=False)
    # display(alerts)
else:
    print("Optional alert report disabled.")

---
## 10. 최종 관측 보고서 (노트북에 직접 작성)

### A. 문제 정의와 데이터
- 객체 탐지 자체가 아니라 어떤 **관측 업무**를 자동화하는가?
- 클래스 4개, 이미지 수, 정답 좌표계, 원래 train/val/test 분할은?

### B. 전이학습 실험
- `frozen_10`, `full_0` (선택 `partial_5`)의 공통조건과 차이·가설은?
- 실험별 mAP50-95는? **공통 임곗값**에서의 Count MAE는? 두 순위가 일치하는가?
- 검증을 사용하여 **최종 모델과 임곗값**을 어떤 절차로 선택했는가?

### C. 실제 활용 결과
- **최종 테스트** mAP50, mAP50-95, 사진·클래스별 Count MAE를 제시하라.
- 객체별 CSV 및 사진별 CSV에서 무엇을 읽을 수 있는가? 4종의 탐지 객체 수·출현 사진 수·출현 사진 비율을 제시하라.
- 세 가지 오류 사례를 **사진과 수치**로 설명하고 관측 시스템에 미치는 영향을 논의하라.
- 새 사진을 사용한 경우 결과물을 제시하라. (정답이 없으면 성능을 단정하지 말 것.)

### D. 결론·한계 및 AI 활용
- **연구 질문:** ‘탐지 성능이 높아지면 개체 수 집계도 정확해지는가?’에 **본 실험 범위에서** 답하라.
- 실제 개체군 규모·이동 경로·시간별 활동 패턴을 이 데이터로 판단할 수 없는 이유는?
- AI 도움을 받은 작업 중 **3개 사례**에 대해 자신의 검증 방법과 수정 결과를 적어라.
- 자신이 직접 학습한 모델과 교수자가 제공한 가중치를 구분하라.

**최종 결론 (실제 수치에 근거한 8~12문장):**

> TODO ★

**AI 활용·검증 기록 1:**

> 질문 / 제안 / 검증 / 나의 수정 및 판단

**AI 활용·검증 기록 2:**

> 질문 / 제안 / 검증 / 나의 수정 및 판단

**AI 활용·검증 기록 3:**

> 질문 / 제안 / 검증 / 나의 수정 및 판단

## 제출 전 자기점검

- [ ] `TODO ★ 1~9`의 필수 코드·실험·해석을 완성하고 **단위 테스트를 통과**했다.
- [ ] 1,504장과 **원본 train/val/test 분할**을 확인했다.
- [ ] **COCO 사전훈련 모델**과 **4클래스 전이학습 모델**의 차이를 정확히 설명했다.
- [ ] 동일 사전훈련 체크포인트의 두 동결 전략을 비교했다(또는 교수자 제공 체크포인트 출처 표시).
- [ ] 두 모델 이상의 **val mAP**와 공통 임곗값의 **val Count MAE**를 모두 측정했다.
- [ ] 모델 선택 규칙·임곗값 선택은 **검증셋**에서만 적용했다.
- [ ] 모델·임곗값 확정 후 **테스트셋을 한 번만** 평가했다.
- [ ] 관측 CSV 2종, 클래스별 객체 수·출현 사진 수·출현 사진 비율 그래프/표를 만들었다.
- [ ] 오류 사례 3장 이상을 **시각적 근거**와 함께 해석했다.
- [ ] 새 사진에서 관측 시스템을 실행하거나 실행 제한/대안과 이유를 기록했다.
- [ ] 사진별 탐지 수와 실제 개체군 크기를 **혼동하지 않았다**.
- [ ] AI 활용·검증 기록 3개, 자신이 수행한 학습/제공 가중치 사용 여부를 표시했다.

**참고 링크**
- [African Wildlife 데이터셋](https://docs.ultralytics.com/datasets/detect/african-wildlife/)
- [YOLO11](https://docs.ultralytics.com/models/yolo11/)
- [YOLO 학습/동결 옵션](https://docs.ultralytics.com/modes/train/)
- [YOLO 평가](https://docs.ultralytics.com/modes/val/)
- [YOLO 추론 결과](https://docs.ultralytics.com/modes/predict/)

*데이터 라이선스, 이미지 재사용 조건 및 Ultralytics 소프트웨어 라이선스는 배포·상업 사용 전에 확인해야 한다.*